In [1]:
import numpy as np
import pandas as pd
import hssm
import arviz as az
import sqlite3
from datetime import datetime

/Users/javierrojas/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
df_hssm = pd.read_csv('data/hssm_exp4_data_ab.csv')

In [3]:
df_hssm[df_hssm['participant_id'] == 708]

,response,rt,participant_id,ab_nominal
0,0,1.4546,708,0.05
1,0,2.0167,708,0.05
2,1,0.9597,708,0.05
3,1,0.9102,708,0.05
4,0,1.0218,708,0.05
...,...,...,...,...
81,1,1.9228,708,10.00
82,1,0.9431,708,10.00
83,1,2.2269,708,10.00
84,1,1.1915,708,10.00


In [5]:
df_hssm['ab_nominal_binary'] = (df_hssm['ab_nominal'] == 10).astype(int)

In [19]:
df_hssm[df_hssm.ab_nominal_binary == 1].count()/df_hssm.count()

response             0.485385
rt                   0.485385
participant_id       0.485385
ab_nominal           0.485385
ab_nominal_binary    0.485385
dtype: float64

In [ ]:
def get_fitted_participants(db_path, table_name):
    with sqlite3.connect(db_path) as conn:
        try:
            q = f"SELECT DISTINCT participant_id FROM {table_name}"
            return set(pd.read_sql(q, conn)['participant_id'])
        except Exception:
            return set()


def write_summary_to_sql(df, db_path, table_name):
    df = df.copy()
    df["timestamp"] = datetime.now().isoformat()

    with sqlite3.connect(db_path) as conn:
        df.to_sql(table_name, conn, if_exists="append", index=False)


In [ ]:
def fit_hssm_mod_th_v_single(
    df, participant_id, participant_column,
    predictor='ab_nominal', use_log=False
):
    df = df.copy()

    df['X'] = (df[predictor] == 10).astype(int)

    df_sub = (
        df[df[participant_column] == participant_id]
        .drop(columns=[participant_column])
    )

    print(f"___Participant {participant_id} | TH + V ___")
    print("Median RT =", np.median(df_sub['rt']))
    print("N trials =", len(df_sub))

    a_prior = {
        "Intercept": {"name": "Normal", "mu": 1.35, "sigma": 0.35},
        "X": {"name": "Normal", "mu": 0.0, "sigma": 0.25},
    }
    v_prior = {
        "Intercept": {"name": "Normal", "mu": 0.45, "sigma": 0.22},
        "X": {"name": "Normal", "mu": 0.0, "sigma": 0.15},
    }

    model = hssm.HSSM(
        data=df_sub,
        model="ddm",
        include=[
            {"name": "a", "formula": "a ~ 1 + X", "prior": a_prior},
            {"name": "v", "formula": "v ~ 1 + X", "prior": v_prior},
        ],
    )

    idata = model.sample(
        cores=3,
        chains=3,
        draws=300,
        tune=1000,
        progressbar=True,
        target_accept=0.99,
    )

    summary_df = (
        az.summary(idata)
        .reset_index()
        .rename(columns={"index": "param"})
    )
    summary_df["participant_id"] = participant_id

    return summary_df


In [ ]:
def fit_hssm_mod_th_single(
    df, participant_id, participant_column,
    predictor='ab_nominal', use_log=False
):
    df = df.copy()

    df['X'] = (df[predictor] == 10).astype(int)

    df_sub = (
        df[df[participant_column] == participant_id]
        .drop(columns=[participant_column])
    )

    print(f"___Participant {participant_id} | TH only ___")
    print("Median RT =", np.median(df_sub['rt']))
    print("N trials =", len(df_sub))

    a_prior = {
        "Intercept": {"name": "Normal", "mu": 1.35, "sigma": 0.35},
        "X": {"name": "Normal", "mu": 0.0, "sigma": 0.25},
    }
    v_prior = {
        "Intercept": {"name": "Normal", "mu": 0.45, "sigma": 0.22},
    }

    model = hssm.HSSM(
        data=df_sub,
        model="ddm",
        include=[
            {"name": "a", "formula": "a ~ 1 + X", "prior": a_prior},
            {"name": "v", "formula": "v ~ 1", "prior": v_prior},
        ],
    )

    idata = model.sample(
        cores=3,
        chains=3,
        draws=300,
        tune=1000,
        progressbar=True,
        target_accept=0.99,
    )

    summary_df = (
        az.summary(idata)
        .reset_index()
        .rename(columns={"index": "param"})
    )
    summary_df["participant_id"] = participant_id

    return summary_df


In [ ]:
def fit_hssm_mod_v_single(
    df, participant_id, participant_column,
    predictor='ab_nominal', use_log=False
):
    df = df.copy()

    df['X'] = (df[predictor] == 10).astype("float64")

    df_sub = (
        df[df[participant_column] == participant_id]
        .drop(columns=[participant_column])
    )

    print(f"___Participant {participant_id} | V only ___")
    print("Median RT =", np.median(df_sub['rt']))
    print("N trials =", len(df_sub))

    a_prior = {
        "Intercept": {"name": "Normal", "mu": 1.35, "sigma": 0.35},
    }
    v_prior = {
        "Intercept": {"name": "Normal", "mu": 0.45, "sigma": 0.22},
        "X": {"name": "Normal", "mu": 0.0, "sigma": 0.15},
    }

    model = hssm.HSSM(
        data=df_sub,
        model="ddm",
        include=[
            {"name": "a", "formula": "a ~ 1", "prior": a_prior},
            {"name": "v", "formula": "v ~ 1 + X", "prior": v_prior},
        ],
    )

    idata = model.sample(
        cores=3,
        chains=3,
        draws=300,
        tune=1000,
        progressbar=True,
        target_accept=0.99,
    )

    summary_df = (
        az.summary(idata)
        .reset_index()
        .rename(columns={"index": "param"})
    )
    summary_df["participant_id"] = participant_id

    return summary_df


In [ ]:
def fit_hssm_mod_v(
    df, participant_id, participant_column
):
    df = df.copy()

    df['X'] = df['ab_nominal_binary'].astype("float64")

    df = df.astype(np.float32)

    df_sub = (
        df[df[participant_column] == participant_id]
        .drop(columns=[participant_column])
    )

    print(f"___Participant {participant_id} | V only ___")
    print("Median RT =", np.median(df_sub['rt']))
    print("N trials =", len(df_sub))

    a_prior = {
        "Intercept": {"name": "Normal", "mu": 1.35, "sigma": 0.35},
    }
    v_prior = {
        "Intercept": {"name": "Normal", "mu": 0.45, "sigma": 0.22},
        "X": {"name": "Normal", "mu": 0.0, "sigma": 0.15},
    }

    model = hssm.HSSM(
        data=df_sub,
        model="ddm",
        include=[
            {"name": "a", "formula": "a ~ 1", "prior": a_prior},
            {"name": "v", "formula": "v ~ 1 + X", "prior": v_prior},
        ],
    )

    idata = model.sample(
        cores=3,
        chains=3,
        draws=300,
        tune=1000,
        progressbar=True,
        target_accept=0.99,
    )

    summary_df = (
        az.summary(idata)
        .reset_index()
        .rename(columns={"index": "param"})
    )
    summary_df["participant_id"] = participant_id

    return summary_df

In [ ]:
def run_sequential_fits(
    df_hssm,
    participant_column,
    db_path,
    predictor='ab_nominal',
    use_log=False,
    max_participants=10,
    model_name=None,
):
    participants = df_hssm[participant_column].unique()

    models = {
        "ddm_mod_th":   fit_hssm_mod_th_single,
        "ddm_mod_v":    fit_hssm_mod_v_single,
    }

    # If model_name specified, only run that model
    if model_name:
        models = {model_name: models[model_name]}

    for table_name, fit_func in models.items():
        print(f"\n===== Running model: {table_name} =====")

        fitted = get_fitted_participants(db_path, table_name)
        remaining = [p for p in participants if p not in fitted]

        print(f"{len(remaining)} participants remaining")

        fitted_count = 0

        for i, pid in enumerate(remaining, 1):
            if fitted_count >= max_participants:
                print(
                    f"\n⏸️  Reached {max_participants} participants "
                    f"for model {table_name}. Re-run to continue."
                )
                break

            print(f"\n--- {table_name}: participant {pid} ({i}/{len(remaining)}) ---")

            try:
                summary_df = fit_func(
                    df=df_hssm,
                    participant_id=pid,
                    participant_column=participant_column,
                    predictor=predictor,
                    use_log=use_log,
                )

                write_summary_to_sql(
                    summary_df,
                    db_path=db_path,
                    table_name=table_name,
                )

                fitted_count += 1

            except Exception as e:
                print(f"❌ Failed participant {pid}: {e}")
                continue


In [ ]:
def run_ddm_mod_v_sequential(
    df_hssm,
    participant_column,
    db_path,
    predictor="ab_nominal",
    use_log=False,
):
    participants = np.sort(df_hssm[participant_column].unique())
    fitted = set(get_fitted_participants(db_path, "ddm_mod_v"))
    remaining = [p for p in participants if p not in fitted]

    print(f"{len(remaining)} participants remaining")

    for i, pid in enumerate(remaining, 1):
        print(f"\n--- ddm_mod_v: participant {pid} ({i}/{len(remaining)}) ---")

        try:
            summary_df = fit_hssm_mod_v_single(
                df=df_hssm,
                participant_id=pid,
                participant_column=participant_column,
                predictor=predictor,
                use_log=use_log,
            )

            write_summary_to_sql(
                summary_df,
                db_path=db_path,
                table_name="ddm_mod_v",
            )

        except Exception as e:
            print(f"❌ Failed participant {pid}: {e}")


In [ ]:
# Fit the next 20 for ddm_mod_th
run_sequential_fits(
    df_hssm=df_hssm,
    participant_column="participant_id",
    db_path="hssm_fits.sqlite",
    predictor="ab_nominal",
    use_log=False,
    max_participants=5,
    model_name="ddm_mod_v",
)

In [ ]:
import sqlite3
import pandas as pd

db_path = "hssm_fits.sqlite"
with sqlite3.connect(db_path) as conn:
    # Get all tables that exist
    tables = pd.read_sql(
        "SELECT name FROM sqlite_master WHERE type='table';",
        conn
    )
    print("Tables in database:")
    print(tables)

In [ ]:
with sqlite3.connect(db_path) as conn:
    # Read a specific table, e.g., 'ddm_mod_th_v'
    df_ddm_mod_th_v = pd.read_sql("SELECT * FROM ddm_mod_th;", conn)
    print("\nData from ddm_mod_th table:")
    n_df = pd.DataFrame(df_ddm_mod_th_v)

In [ ]:
len(n_df.participant_id.unique())

In [ ]:
n_df[n_df['participant_id'] == 708]